The URL of interest looks like:

https://www.ncbi.nlm.nih.gov/nuccore/?term=SMAD3%5BGene%20Name%5D%20AND%20Homo%20sapiens%5BOrganism%5D%20AND%20srcdb_refseq%5BPROP%5D

In [1]:
URL = "https://www.ncbi.nlm.nih.gov/nuccore/?term=SMAD3%5BGene%20Name%5D%20AND%20Homo%20sapiens%5BOrganism%5D%20AND%20srcdb_refseq%5BPROP%5D"


In [2]:
# Fetch a web page with urllib and return its HTML source
import urllib.request
import urllib.error
from typing import Optional


def fetch_page(url: str, timeout: float = 10.0) -> str:
    """Retrieve the web page at the given URL and return decoded HTML.

    - Uses a browser-like User-Agent to avoid 403 blocks.
    - Decodes using the charset from Content-Type if present, else UTF-8.
    - Raises urllib.error.URLError/HTTPError on network or HTTP issues.
    """
    req = urllib.request.Request(
        url,
        headers={
            "User-Agent": "Mozilla/5.0 (X11; Linux) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120 Safari/537.36",
            "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
            "Accept-Language": "en-US,en;q=0.9",
        },
    )
    with urllib.request.urlopen(req, timeout=timeout) as resp:
        # Prefer encoding from headers, fall back to UTF-8
        charset: Optional[str] = None
        try:
            charset = resp.headers.get_content_charset()
        except Exception:
            charset = None
        encoding = charset or "utf-8"
        data = resp.read()
        return data.decode(encoding, errors="replace")


In [3]:
# Use the function to fetch the page (preview only)
# Note: This performs a network request to NCBI.
try:
    page_html = fetch_page(URL)
    print("Downloaded characters:", len(page_html))
    print("Preview:\n", page_html[:500])
except urllib.error.HTTPError as e:
    print("HTTPError:", e.code, e.reason)
except urllib.error.URLError as e:
    print("URLError:", e.reason)


Downloaded characters: 136858
Preview:
 <?xml version="1.0" encoding="utf-8"?>
<!DOCTYPE html PUBLIC "-//W3C//DTD XHTML 1.0 Transitional//EN" "http://www.w3.org/TR/xhtml1/DTD/xhtml1-transitional.dtd">
<html xmlns="http://www.w3.org/1999/xhtml" lang="en" xml:lang="en">
    <head xmlns:xi="http://www.w3.org/2001/XInclude"><meta http-equiv="Content-Type" content="text/html; charset=utf-8" />
    <!-- meta -->
    <meta name="robots" content="noindex,follow,noarchive" />
<meta name="ncbi_app" content="entrez" /><meta name="ncbi_db" conten


In [4]:
page_html

'<?xml version="1.0" encoding="utf-8"?>\n<!DOCTYPE html PUBLIC "-//W3C//DTD XHTML 1.0 Transitional//EN" "http://www.w3.org/TR/xhtml1/DTD/xhtml1-transitional.dtd">\n<html xmlns="http://www.w3.org/1999/xhtml" lang="en" xml:lang="en">\n    <head xmlns:xi="http://www.w3.org/2001/XInclude"><meta http-equiv="Content-Type" content="text/html; charset=utf-8" />\n    <!-- meta -->\n    <meta name="robots" content="noindex,follow,noarchive" />\n<meta name="ncbi_app" content="entrez" /><meta name="ncbi_db" content="nuccore" /><meta name="ncbi_term" content="smad3[gene name] and homo sapiens[organism] and srcdb_refseq[prop]" /><meta name="ncbi_report" content="docsum" /><meta name="ncbi_format" content="html" /><meta name="ncbi_pagesize" content="20" /><meta name="ncbi_sortorder" content="default" /><meta name="ncbi_pageno" content="1" /><meta name="ncbi_resultcount" content="16" /><meta name="ncbi_op" content="search" /><meta name="ncbi_pdid" content="docsum" /><meta name="ncbi_sessionid" content

In [5]:
# Extract RefSeq accessions from text/HTML and categorize
import re
from typing import Dict, Set, List

REFSEQ_PREFIXES = (
    "NM", "NR",  # curated transcripts
    "XM", "XR",  # model transcripts
    "NP",         # curated proteins
    "XP",         # model proteins
    "WP", "YP",  # nonredundant / historical proteins
    "NC", "NG", "NT", "NW",  # genomic sequences
)

# Regex for accessions like NM_005902.4 (version optional)
REFSEQ_REGEX = re.compile(r"\b(?:NM|NR|XM|XR|NP|XP|WP|YP|NC|NG|NT|NW)_\d+(?:\.\d+)?\b")


def extract_refseq_ids(text: str) -> Dict[str, List[str]]:
    """Find RefSeq accessions in the given text and return categorized lists.

    Returns a dict with keys:
    - all: all unique accessions found (sorted)
    - nucleotide_curated: NM_, NR_
    - nucleotide_model: XM_, XR_
    - protein_curated: NP_
    - protein_model: XP_
    - protein_cluster: WP_, YP_
    - genomic: NC_, NG_, NT_, NW_
    """
    matches = set(REFSEQ_REGEX.findall(text or ""))

    def pick(prefixes: Set[str]) -> List[str]:
        return sorted(x for x in matches if any(x.startswith(p + "_") for p in prefixes))

    result: Dict[str, List[str]] = {
        "all": sorted(matches),
        "nucleotide_curated": pick({"NM", "NR"}),
        "nucleotide_model": pick({"XM", "XR"}),
        "protein_curated": pick({"NP"}),
        "protein_model": pick({"XP"}),
        "protein_cluster": pick({"WP", "YP"}),
        "genomic": pick({"NC", "NG", "NT", "NW"}),
    }
    return result

# Run on the fetched page and show a compact summary
ids = extract_refseq_ids(page_html)
print("Total unique RefSeq IDs:", len(ids["all"]))
for k in [
    "nucleotide_curated", "nucleotide_model", "protein_curated",
    "protein_model", "protein_cluster", "genomic"
]:
    bucket = ids[k]
    print(f"{k}: {len(bucket)}")

# Show a few examples
for k in ["nucleotide_curated", "protein_curated", "genomic"]:
    if ids[k]:
        print(f"\nExamples from {k}:")
        print("\n".join(ids[k][:5]))


Total unique RefSeq IDs: 32
nucleotide_curated: 22
nucleotide_model: 4
protein_curated: 0
protein_model: 0
protein_cluster: 0
genomic: 6

Examples from nucleotide_curated:
NM_001145102
NM_001145102.2
NM_001145103
NM_001145103.2
NM_001145104

Examples from genomic:
NC_000015
NC_000015.10
NC_060939
NC_060939.1
NG_011990
